<a href="https://colab.research.google.com/github/pmeyssonnier/pv-explorer-app/blob/main/PV_Schaerbeek_scrapper_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ PV Schaerbeek — pipeline d'extraction (v3, sécurisé & économe)

Notebook réorganisé **par phases**. Règle d'or : **on paie le LLM le moins possible**.

| Phase | Coût | Quand l'exécuter |
|---|---|---|
| 0 · Setup | gratuit | à chaque session |
| 1 · Audit complétude | **gratuit** (0 appel LLM) | à chaque session, en 1er |
| 2 · Re-extraction **ciblée** | faible (≈ trous réels) | **le mode normal** |
| 3 · Pipeline complet | **cher** | ⚠️ manuel, exceptionnel |
| 4 · Valider + copier le JSON | gratuit | avant de committer |
| 5 · Commit → `main` | gratuit | quand la base est bonne |
| 6 · Indexation Pinecone | quota embeddings | **quand tu auras du crédit** |

### 🔐 Secrets — AUCUNE clé en dur
Toutes les clés viennent du **gestionnaire de secrets Colab** (icône 🔑 à gauche).
À créer une seule fois, avec *Accès au notebook* activé :
- `ANTHROPIC_API_KEY`
- `GITHUB_PAT`
- `PINECONE_API_KEY`

> ⚠️ Les anciennes clé Anthropic et PAT GitHub qui étaient écrites en dur dans
> la v2 sont **compromises** : révoque-les et régénère-les avant d'aller plus loin.

## Phase 0 — Setup

In [19]:
# 0.1 — Dépendances
!pip install -q requests beautifulsoup4 tqdm pdfplumber anthropic

In [ ]:
# 0.2 — Drive + dépôt calé EXACTEMENT sur origin/main (aucune fusion, pas de divergence)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('/content/pv-explorer-app'):
    !git clone https://github.com/pmeyssonnier/pv-explorer-app.git /content/pv-explorer-app
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

# État de la base versionnée dans le dépôt
!python -c "import json; d=json.load(open('backend/pv_conseil_schaerbeek.json')); print(len(d['seances']),'séances,', sum(len(s['points']) for s in d['seances']),'points')"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/pv-explorer-app
From https://github.com/pmeyssonnier/pv-explorer-app
 * branch            main       -> FETCH_HEAD
Branch 'main' set up to track remote branch 'main' from 'origin'.
Reset branch 'main'
Your branch is up to date with 'origin/main'.
127 séances, 7309 points


In [20]:
# 0.3 — 🔐 Secrets depuis Colab (jamais écrits dans le notebook)
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ ANTHROPIC_API_KEY chargée depuis les secrets Colab')
except Exception as e:
    # Repli hors-Colab : saisie masquée, toujours pas de clé en dur
    if not os.environ.get('ANTHROPIC_API_KEY'):
        import getpass
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Clé Anthropic : ')
    print('⚠️ userdata indisponible (', type(e).__name__, ') → getpass utilisé')

✅ ANTHROPIC_API_KEY chargée depuis les secrets Colab


In [ ]:
# 0.4 — Imports pipeline + garde-fous
%cd /content/pv-explorer-app/pipeline

# ⚠️ Purge le cache d'imports : force la relecture des .py FRAÎCHEMENT checkout
# (sinon Python garderait la version importée lors d'un run précédent, malgré le
#  git checkout de la Phase 0.2). Garantit que les imports = code de main.
import sys
for _m in ['pv_extraction_pipeline', 'audit_completeness', 'reextract_targeted']:
    sys.modules.pop(_m, None)

from pv_extraction_pipeline import (
    CONFIG, run_pipeline, validate_database, stats_summary, export_csv, get_pdf_list
)

print('MODEL      =', CONFIG['MODEL'])
print('MAX_TOKENS =', CONFIG['MAX_TOKENS'], '| CHUNK_SIZE =', CONFIG['CHUNK_SIZE'])
assert CONFIG['MAX_TOKENS'] == 8192 and CONFIG['CHUNK_SIZE'] == 8, \
    '❌ Ancien code en mémoire → Exécution > Redémarrer la session, puis relance Phase 0'

pdfs = get_pdf_list(CONFIG['INPUT_DIR'])
print(f'✅ {len(pdfs)} PDF prêts dans {CONFIG["INPUT_DIR"]}')

/content/pv-explorer-app/pipeline
MODEL      = claude-haiku-4-5-20251001
MAX_TOKENS = 8192 | CHUNK_SIZE = 8


2026-08-18 23:18:13,252 [INFO] PDF trouvés : 190 fichiers dans /content/drive/MyDrive/PV_Schaerbeek/input
INFO:pv_pipeline:PDF trouvés : 190 fichiers dans /content/drive/MyDrive/PV_Schaerbeek/input


✅ 190 PDF prêts dans /content/drive/MyDrive/PV_Schaerbeek/input


In [ ]:
# 0.5 — (optionnel, gratuit) vérifie que 2 PDF s'ouvrent, SANS appeler l'API
run_pipeline(max_files=2, dry_run=True)

## Phase 1 — Audit de complétude (GRATUIT, 0 appel LLM)

Compare, séance par séance, les points **attendus** (regex `SP n.-` sur le PDF)
aux points **présents** dans la base. Révèle les trous **avant** de payer quoi que ce soit.

In [ ]:
from pv_extraction_pipeline import load_database, CONFIG
from audit_completeness import audit_completeness, print_audit

db = load_database()
report = audit_completeness(db, CONFIG['INPUT_DIR'])   # aucun appel Claude
summary = print_audit(report)                          # séances incomplètes + SP manqués

2026-08-18 23:18:18,644 [INFO] Base existante chargée : 134 séances
INFO:pv_pipeline:Base existante chargée : 134 séances
2026-08-18 23:18:25,875 [INFO]   PDF extrait : 32 pages — 91,870 chars
INFO:pv_pipeline:  PDF extrait : 32 pages — 91,870 chars
2026-08-18 23:18:34,659 [INFO]   PDF extrait : 41 pages — 106,925 chars
INFO:pv_pipeline:  PDF extrait : 41 pages — 106,925 chars
2026-08-18 23:18:51,362 [INFO]   PDF extrait : 103 pages — 315,388 chars
INFO:pv_pipeline:  PDF extrait : 103 pages — 315,388 chars
2026-08-18 23:19:12,495 [INFO]   PDF extrait : 124 pages — 393,887 chars
INFO:pv_pipeline:  PDF extrait : 124 pages — 393,887 chars
2026-08-18 23:19:30,279 [INFO]   PDF extrait : 106 pages — 369,243 chars
INFO:pv_pipeline:  PDF extrait : 106 pages — 369,243 chars
2026-08-18 23:19:42,010 [INFO]   PDF extrait : 68 pages — 215,550 chars
INFO:pv_pipeline:  PDF extrait : 68 pages — 215,550 chars
2026-08-18 23:19:55,680 [INFO]   PDF extrait : 89 pages — 293,645 chars
INFO:pv_pipeline:  PDF


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  AUDIT DE COMPLÉTUDE (hors-ligne, sans LLM)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Séances auditées         : 134
  Points attendus (regex)  : 7796
  Points présents (base)   : 7830
  Séances incomplètes      : 14
    ⚠ 2013-09-25 (pv_2013.09.25.pdf) : 111/112 — SP manquants [98] (pages [185])
    ⚠ 2014-06-25 (pv_2014.06.25_sp.pdf) : 55/56 — SP manquants [19] (pages [48])
    ⚠ 2015-09-23 (pv_2015.09.23_sp.pdf) : 129/131 — SP manquants [85, 86] (pages [96])
    ⚠ 2017-05-31 (PV Conseil 2017.05.31.pdf) : 66/67 — SP manquants [64] (pages [76])
    ⚠ 2017-10-25 (PV Conseil 2017.10.25.pdf) : 80/81 — SP manquants [37] (pages [56])
    ⚠ 2018-03-28 (pv_conseil_2018.03.28_sp.pdf) : 47/48 — SP manquants [34] (pages [61])
    ⚠ 2019-05-08 (pv_conseil_2019.05.08_sp.pdf) : 60/61 — SP manquants [71] (pages [60])
    ⚠ 2022-06-01 (pv_conseil_2022.06.01_sp (1).pdf) : 71/72 — SP manquants [36] (pages [40])
    ⚠ 2022-06-2

## Phase 2 — Re-extraction CIBLÉE  ✅ *mode normal*

Ne relance le LLM **que** sur les séances à trous réels (et, en interne, seulement
sur les **pages manquantes** via la récupération page-par-page). `min_missing=2`
ignore les off-by-one (« 87/88 » = souvent un faux positif regex, pas un vrai manque).

👉 C'est cette phase qu'on exécute au quotidien — **pas** la Phase 3.

In [ ]:
from reextract_targeted import targets_from_audit, reextract_seances

cibles = targets_from_audit(report, min_missing=2)   # ne garde que les vrais trous
print('Cibles :', cibles)

reextract_seances(cibles)    # re-extrait + fusionne + sauvegarde (n'écrase rien d'autre)

# Re-audit pour confirmer que les trous sont comblés
report2 = audit_completeness(load_database(), CONFIG['INPUT_DIR'])
print_audit(report2)

In [ ]:
# (optionnel) cibler manuellement 1-2 séances précises, par date ou nom de PDF
# reextract_seances(['2016-11-30', '2017-04-26'])   # les 2 plus gros gains

## Phase 3 — Pipeline complet  ⚠️ *CHER — ne pas exécuter par défaut*

À n'utiliser **que** pour un premier remplissage massif ou une refonte totale.
Ces cellules sont volontairement **désactivées** (code en commentaire) pour éviter
une facture inutile alors que la base est déjà à ~99,7 % complète.

- `run_pipeline()` : scanne **tous** les PDF (le cache SHA-256 + `progress.json`
  protègent, mais ça reste à éviter en routine).
- `run_pipeline(year=Y, force_reprocess=True)` : **ignore le cache** et re-facture
  **toute** l'année Y. Réserve-le à un PDF précis qu'on sait mal extrait.

> Pour combler des trous, préfère **toujours** la Phase 2 (ciblée).

In [ ]:
# ⚠️ DÉSACTIVÉ. Décommente en pleine conscience du coût (reprend via progress.json).
# db = run_pipeline()

In [ ]:
# ⚠️ DÉSACTIVÉ. force_reprocess=True IGNORE le cache → re-facture toute l'année.
# db = run_pipeline(year=2024, force_reprocess=True)

## Phase 4 — Valider et copier le JSON dans le dépôt

In [ ]:
from pv_extraction_pipeline import load_database, CONFIG
db = load_database()          # recharge la base réellement écrite sur le Drive

validate_database(db)         # titres manquants / SP dupliqués / montants suspects
stats_summary(db)
export_csv(db, '/content/drive/MyDrive/PV_Schaerbeek/pv_all_points.csv')

# Copie le JSON dans le dépôt cloné pour le committer
import shutil
SRC = CONFIG['DB_JSON_PATH']
DST = '/content/pv-explorer-app/backend/pv_conseil_schaerbeek.json'
shutil.copy(SRC, DST)
print('copié →', DST)

In [21]:
# Contrôle final des compteurs avant commit
%cd /content/pv-explorer-app
!python -c "import json; d=json.load(open('backend/pv_conseil_schaerbeek.json')); print(len(d['seances']),'séances,', sum(len(s['points']) for s in d['seances']),'points')"

/content/pv-explorer-app
144 séances, 8718 points


## Phase 5 — Commit du JSON → `main`

Le PAT vient des **secrets Colab** (`GITHUB_PAT`). Il est injecté dans l'URL du remote
*le temps du push*, puis **immédiatement retiré** du remote → jamais persistant, jamais
écrit dans le notebook.

In [ ]:
%cd /content/pv-explorer-app

# Rester calé sur main sans perdre le JSON qu'on vient de copier
!git fetch origin main
!git stash -u    # met de côté le JSON modifié
!git checkout -f -B main origin/main
!git stash pop   # remet le JSON par-dessus la base à jour

/content/pv-explorer-app
From https://github.com/pmeyssonnier/pv-explorer-app
 * branch            main       -> FETCH_HEAD
Saved working directory and index state WIP on main: 536bc58 Ajoute un helper de re-extraction ciblée (complément de l'audit)
Branch 'main' set up to track remote branch 'main' from 'origin'.
Reset branch 'main'
Your branch is up to date with 'origin/main'.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   backend/pv_conseil_schaerbeek.json

no changes added to commit (use "git add" and/or "git commit -a")
Dropped refs/stash@{0} (e23a971905787b5f75e966de530f8a547ec7bef3)


In [ ]:
import os
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    from getpass import getpass
    TOKEN = getpass('GitHub TOKEN : ')

!git config user.email 'pmeyssonnier@gmail.com'
!git config user.name  'pmeyssonnier'
!git add backend/pv_conseil_schaerbeek.json
!git commit -m "data: mise à jour extraction PV Schaerbeek (re-extraction ciblée)"

# push avec le token en clair UNIQUEMENT dans la commande, puis remote nettoyé
!git remote set-url origin https://{TOKEN}@github.com/pmeyssonnier/pv-explorer-app.git
!git push origin main
!git remote set-url origin https://github.com/pmeyssonnier/pv-explorer-app.git
del TOKEN
print('✅ JSON poussé sur main — le backend Render servira la base à jour au prochain déploiement.')

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
remote: Permission to pmeyssonnier/pv-explorer-app.git denied to pmeyssonnier.
fatal: unable to access 'https://github.com/pmeyssonnier/pv-explorer-app.git/': The requested URL returned error: 403
✅ JSON poussé sur main — le backend Render servira la base à jour au prochain déploiement.


## Phase 6 — Indexation Pinecone  🔜 *quand tu auras du crédit*

L'indexation consomme le **quota mensuel d'embeddings** (5 M tokens sur le plan gratuit).
Tant que le quota est épuisé, l'indexation renvoie des `429 RESOURCE_EXHAUSTED` **et**
le chat `/ask` est indisponible (l'embedding de la question passe par le même quota).

**À exécuter seulement une fois le plan Pinecone rechargé/upgradé.**

Astuce coût : commence par `--only-year` sur les années manquantes (2019, 2021, 2022,
2023) plutôt qu'un réindex complet — tu ne re-paies pas les embeddings déjà en place.

In [23]:
!pip install -q pinecone==9.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 8.8 MB/s eta 0:00:00


In [24]:
# Récupérer le dernier index_pv.py + clé Pinecone depuis les secrets
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

import os
try:
    from google.colab import userdata
    os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
    print('✅ PINECONE_API_KEY chargée depuis les secrets Colab')
except Exception:
    from getpass import getpass
    os.environ['PINECONE_API_KEY'] = getpass('PINECONE_API_KEY (pcsk_...) : ')

/content/pv-explorer-app
From https://github.com/pmeyssonnier/pv-explorer-app
 * branch            main       -> FETCH_HEAD
Branch 'main' set up to track remote branch 'main' from 'origin'.
Reset branch 'main'
Your branch is up to date with 'origin/main'.
✅ PINECONE_API_KEY chargée depuis les secrets Colab


In [26]:
# Option A (économe) — n'indexer QUE les années manquantes
%cd /content/pv-explorer-app/backend
!python index_pv.py --input pv_conseil_schaerbeek.json --commune schaerbeek --only-year 2013,2023,2024,2025

/content/pv-explorer-app/backend
════════════════════════════════════════════════════════════
  INDEXATION PV SCHAERBEEK → PINECONE
════════════════════════════════════════════════════════════
✅ Index 'pv-explorer' existe déjà
📄 8718 points chargés depuis pv_conseil_schaerbeek.json (commune par défaut : schaerbeek ; seance['commune'] prioritaire si présent)
🎯 Filtre --only-year [2013, 2023, 2024, 2025] : 8718 → 2884 chunks à (ré)indexer
ℹ️  Vecteurs déjà dans l'index avant ce run : 8424
📤 Indexation de 2884 chunks (batches de 60, cadence initiale ≤ 130,000 tokens/min, auto-adaptative)...
   60/2884 chunks indexés
   120/2884 chunks indexés
   180/2884 chunks indexés
   240/2884 chunks indexés
   300/2884 chunks indexés
   360/2884 chunks indexés
   420/2884 chunks indexés
   480/2884 chunks indexés
   540/2884 chunks indexés
   600/2884 chunks indexés
   660/2884 chunks indexés
   720/2884 chunks indexés
   780/2884 chunks indexés
   840/2884 chunks indexés
   900/2884 chunks indexés
 

In [ ]:
# Option B — réindex complet (⚠️ gros consommateur de quota d'embeddings)
# %cd /content/pv-explorer-app/backend
# !python index_pv.py --input pv_conseil_schaerbeek.json --commune schaerbeek

In [ ]:

%cd /content/pv-explorer-app
# copie le v3 (outputs vides) dans pipeline/ puis :
!git add pipeline/PV_Schaerbeek_scrapper.ipynb
!git commit -m "docs: notebook d'extraction Schaerbeek (v3, secrets externalisés)"
# push via ton github_token

In [22]:
%cd /content/pv-explorer-app
!git log --oneline -1          # doit montrer 72e0b0c
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{TOKEN}@github.com/pmeyssonnier/pv-explorer-app.git
!git push origin main
!git remote set-url origin https://github.com/pmeyssonnier/pv-explorer-app.git
del TOKEN

/content/pv-explorer-app
72e0b0c (HEAD -> main) data: séances manquantes complétées (144 séances / 8718 points)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 107.96 KiB | 443.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
remote: Bypassed rule violations for refs/heads/main:
remote: 
remote: - Required status check "test" is expected.
remote: 
To https://github.com/pmeyssonnier/pv-explorer-app.git
   cad9643..72e0b0c  main -> main
